# Strategy Verification — Clean Auditable Implementation

Independent verification of all strategy computations using 1 year of real SOL/USDC data.
Each strategy is implemented from scratch with step-by-step calculations.

**Fixes applied**: Perp tail closure fees corrected (separate per-leg, not doubled).
**Fee sensitivity**: Tests at 5 different fee rates to assess strategy viability.

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermgauss
from scipy import stats
import requests, time, os, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(42)

# -- Protocol constants --
MARKUP = 1.10
ALPHA = 0.40
BOND_APY = 0.12
JITO_APY = 0.07
SOL_FRACTION = 0.48
PERP_FUNDING_APY = 0.80
PERP_FEE_BPS = 8
PERP_IMPACT_BPS = 3
IV_PREMIUM = 0.25
OPTION_SPREAD_PCT = 0.10
N_LIQ = 10000  # liquidity parameter L
T_WEEK = 7 / 365
BOND_WK = (1 + BOND_APY) ** (1/52) - 1

# -- Width configurations --
WIDTHS = {
    '5pct':  {'label': '+-5%',  'width': 0.05, 'fee': 0.0065, 'fs_bps': 2500},
    '10pct': {'label': '+-10%', 'width': 0.10, 'fee': 0.0045, 'fs_bps': 2000},
}

# -- Fee scenarios --
FEE_SCENARIOS = [0.002, 0.003, 0.0045, 0.0065, 0.010]  # daily rates

print('Constants loaded.')
print(f'  MARKUP={MARKUP}, ALPHA={ALPHA}, PERP_FUNDING={PERP_FUNDING_APY*100:.0f}%APY')
print(f'  Fee scenarios: {[f"{f*100:.1f}%/d" for f in FEE_SCENARIOS]}')

Constants loaded.
  MARKUP=1.1, ALPHA=0.4, PERP_FUNDING=80%APY
  Fee scenarios: ['0.2%/d', '0.3%/d', '0.4%/d', '0.7%/d', '1.0%/d']


In [2]:
import base64
from datetime import datetime, timezone

BIRDEYE_KEY = 'ed577a4a6a4f480fa659b4f18673e4b1'
HELIUS_RPC = 'https://mainnet.helius-rpc.com/?api-key=2ef5fdd0-5c3b-4ae1-a2fc-e12b3fd605e7'
SOL_MINT = 'So11111111111111111111111111111111111111112'
WHIRLPOOL_ACCOUNT = 'Czfq3xZZDmsdGdUyrNLtRhGc47cXcZtLG4crryfu44zE'

# -- Birdeye: 1yr daily in 90-day chunks --
def fetch_birdeye_ohlcv(mint, api_key, days_back=365, chunk_days=90):
    """Fetch daily OHLCV from Birdeye in 90-day chunks."""
    url = 'https://public-api.birdeye.so/defi/ohlcv'
    headers = {'X-API-KEY': api_key, 'Accept': 'application/json'}
    all_candles = []
    now = int(time.time())
    start = now - days_back * 86400
    t = start
    while t < now:
        chunk_end = min(t + chunk_days * 86400, now)
        params = {
            'address': mint,
            'type': '1D',
            'time_from': t,
            'time_to': chunk_end,
        }
        try:
            resp = requests.get(url, headers=headers, params=params, timeout=30)
            items = resp.json().get('data', {}).get('items', [])
            all_candles.extend(items)
            print(f"  Fetched {len(items)} candles from {datetime.fromtimestamp(t, tz=timezone.utc).date()} "
                  f"to {datetime.fromtimestamp(chunk_end, tz=timezone.utc).date()}")
        except Exception as e:
            print(f"  Warning: chunk failed ({e}), continuing...")
        t = chunk_end
        time.sleep(0.3)

    # Deduplicate by timestamp
    seen = set()
    unique = []
    for c in all_candles:
        ts = c.get('unixTime', c.get('time', 0))
        if ts not in seen:
            seen.add(ts)
            unique.append(c)
    unique.sort(key=lambda c: c.get('unixTime', c.get('time', 0)))
    return unique

print("Fetching SOL/USDC daily data from Birdeye...")
raw_candles = fetch_birdeye_ohlcv(SOL_MINT, BIRDEYE_KEY, days_back=365, chunk_days=90)
print(f"Total candles fetched: {len(raw_candles)}")

# Parse close prices
timestamps = []
closes_list = []
for c in raw_candles:
    ts = c.get('unixTime', c.get('time', 0))
    timestamps.append(datetime.fromtimestamp(ts, tz=timezone.utc))
    closes_list.append(float(c.get('c', c.get('close', 0))))

closes = np.array(closes_list)
dates = np.array(timestamps)

# Filter invalid
valid = closes > 0
closes = closes[valid]
dates = dates[valid]

print(f"Valid data points: {len(closes)}")
if len(closes) > 0:
    print(f"Date range: {dates[0].date()} to {dates[-1].date()}")
    print(f"Price range: ${closes.min():.2f} -- ${closes.max():.2f}")

# -- Orca: current price from Whirlpool account --
def fetch_orca_price(rpc_url, account_pubkey):
    """Fetch current SOL/USDC price from Orca Whirlpool on-chain."""
    payload = {
        'jsonrpc': '2.0', 'id': 1, 'method': 'getAccountInfo',
        'params': [account_pubkey, {'encoding': 'base64'}]
    }
    try:
        r = requests.post(rpc_url, json=payload, timeout=15)
        data = r.json()
        account_data = base64.b64decode(data['result']['value']['data'][0])
        sqrt_price_x64 = int.from_bytes(account_data[65:81], 'little')
        price = (sqrt_price_x64 / (2**64))**2 * 1e3
        return price
    except Exception as e:
        print(f"  Warning: Orca fetch failed ({e}), using last Birdeye close")
        return None

current_price_orca = fetch_orca_price(HELIUS_RPC, WHIRLPOOL_ACCOUNT)
if current_price_orca is not None:
    S0 = current_price_orca
    print(f"Current SOL/USDC from Orca Whirlpool: ${S0:.4f}")
else:
    S0 = closes[-1] if len(closes) > 0 else 130.0
    print(f"Using last Birdeye close: ${S0:.4f}")

# Realized vol
log_returns = np.log(closes[1:] / closes[:-1])
realized_vol = float(np.std(log_returns, ddof=1) * np.sqrt(365))
print(f"\nRealized annual vol: {realized_vol*100:.1f}%")
print(f"Reference price S0 = ${S0:.2f}")

Fetching SOL/USDC daily data from Birdeye...


  Fetched 90 candles from 2025-04-12 to 2025-07-11


  Fetched 90 candles from 2025-07-11 to 2025-10-09


  Fetched 90 candles from 2025-10-09 to 2026-01-07


  Fetched 90 candles from 2026-01-07 to 2026-04-07


  Fetched 5 candles from 2026-04-07 to 2026-04-12


Total candles fetched: 365
Valid data points: 365
Date range: 2025-04-13 to 2026-04-12
Price range: $77.88 -- $247.54
Current SOL/USDC from Orca Whirlpool: $81.6403

Realized annual vol: 73.7%
Reference price S0 = $81.64


In [3]:
def cl_value_vec(S, L, p_l, p_u):
    S = np.atleast_1d(np.asarray(S, float))
    spl, spu = np.sqrt(p_l), np.sqrt(p_u)
    sp = np.sqrt(np.clip(S, 1e-10, None))
    bl = S <= p_l; ab = S >= p_u
    a = np.where(bl, L*(spu-spl)/(spl*spu), np.where(ab, 0., L*(spu-sp)/(sp*spu)))
    b = np.where(bl, 0., np.where(ab, L*(spu-spl), L*(sp-spl)))
    return a*S + b

def corridor_payoff_vec(S_T, S0, B, Cap, L, p_l, p_u):
    S_T = np.atleast_1d(np.asarray(S_T, float))
    V0 = cl_value_vec(S0, L, p_l, p_u)
    V_eff = cl_value_vec(np.maximum(S_T, B), L, p_l, p_u)
    return np.minimum(Cap, np.maximum(0., np.where(S_T >= S0, 0., V0 - V_eff)))

def make_position(S0, width):
    p_l = S0 * (1 - width); p_u = S0 * (1 + width)
    V0 = float(cl_value_vec(np.array([S0]), N_LIQ, p_l, p_u)[0])
    V_B = float(cl_value_vec(np.array([p_l]), N_LIQ, p_l, p_u)[0])
    return {'S0': S0, 'p_l': p_l, 'p_u': p_u, 'L': N_LIQ,
            'V0': V0, 'barrier': p_l, 'nat_cap': V0 - V_B}

def fv_quadrature(S0, sigma, B, Cap, L, p_l, p_u):
    nodes, weights = hermgauss(128)
    S_T = S0 * np.exp(-sigma**2/2*T_WEEK + sigma*np.sqrt(T_WEEK)*nodes*np.sqrt(2))
    payoffs = corridor_payoff_vec(S_T, S0, B, Cap, L, p_l, p_u)
    return max(0, float(np.sum(weights * payoffs) / np.sqrt(np.pi)))

def bs_put(S, K, sig, T):
    if T <= 0 or sig <= 0: return max(0., K-S)
    d1 = (np.log(S/K)+(sig**2/2)*T)/(sig*np.sqrt(T))
    d2 = d1 - sig*np.sqrt(T)
    return K*stats.norm.cdf(-d2) - S*stats.norm.cdf(-d1)

def cl_delta(S, L, p_l, p_u):
    return float((cl_value_vec(S+0.01, L, p_l, p_u) - cl_value_vec(S-0.01, L, p_l, p_u)) / 0.02)

print('Core functions defined.')

Core functions defined.


In [4]:
print('='*80)
print('MANUAL SINGLE-WEEK WALKTHROUGH')
print('='*80)

for wk_label, wcfg in WIDTHS.items():
    w = wcfg['width']; daily_fee = wcfg['fee']
    fs_frac = wcfg['fs_bps'] / 10000

    # Pick week 5 (index 35-42)
    si = 35; ei = 42
    S_s = closes[si]; S_e = closes[ei]
    week_prices = closes[si:ei+1]

    print(f'\n--- {wcfg["label"]} Width, Week 5 ---')
    print(f'Entry: ${S_s:.2f}, Exit: ${S_e:.2f}, Change: {(S_e/S_s-1)*100:+.2f}%')

    pos = make_position(S_s, w)
    V0 = pos['V0']; nat_cap = pos['nat_cap']
    V_end = float(cl_value_vec(np.array([S_e]), N_LIQ, pos['p_l'], pos['p_u'])[0])
    IL = V_end - V0

    print(f'Position: V0=${V0:.2f}, V_end=${V_end:.2f}, IL=${IL:.2f} ({IL/V0*100:+.2f}%)')
    print(f'Range: [${pos["p_l"]:.2f}, ${pos["p_u"]:.2f}], Barrier=${pos["barrier"]:.2f}')
    print(f'Natural cap: ${nat_cap:.2f} ({nat_cap/V0*100:.1f}% of V0)')

    # In-range fraction
    in_rng = np.mean((week_prices[1:] >= pos['p_l']) & (week_prices[1:] <= pos['p_u']))
    fees = V0 * daily_fee * 7 * (in_rng * 0.95 + 0.05)
    print(f'In-range: {in_rng*100:.0f}%, Fees: ${fees:.2f} ({fees/V0*100:.2f}%)')

    # Trailing vol
    lr = np.diff(np.log(closes[max(0,si-30):si+1]))
    sigma = float(np.std(lr, ddof=1) * np.sqrt(365))
    print(f'Trailing 30d vol: {sigma*100:.1f}%')

    # Fair value and payout
    fv = fv_quadrature(S_s, sigma, pos['barrier'], nat_cap, N_LIQ, pos['p_l'], pos['p_u'])
    pay = float(corridor_payoff_vec(np.array([S_e]), S_s, pos['barrier'], nat_cap, N_LIQ, pos['p_l'], pos['p_u'])[0])
    print(f'Fair value: ${fv:.2f}, Payout: ${pay:.2f}')

    # Strategy returns
    print(f'\nStrategy returns (% of V0):')

    # 1. Bond
    print(f'  Bond:        {BOND_WK*100:+.3f}%')

    # 2. Plain LP
    plain = (IL + fees) / V0
    print(f'  Plain LP:    {plain*100:+.3f}%  (IL={IL/V0*100:+.2f}% + fees={fees/V0*100:+.2f}%)')

    # 3. Corridor (fixed)
    prem_fixed = fv * MARKUP
    eff_prem = max(0, prem_fixed - fees * fs_frac)
    corr_fixed = (IL + fees + pay - eff_prem) / V0
    print(f'  Corridor(fx):{corr_fixed*100:+.3f}%  (prem=${prem_fixed:.2f}, eff=${eff_prem:.2f}, pay=${pay:.2f})')

    # 4. Corridor (two-part)
    wf_exp = V0 * daily_fee * 7
    beta = max(0, (MARKUP - ALPHA) * fv / max(wf_exp, 1e-10))
    lr7 = np.diff(np.log(closes[max(0,si-7):si+1]))
    s7 = float(np.std(lr7, ddof=1) * np.sqrt(365)) if len(lr7) > 1 else sigma
    vi = np.clip(s7 / max(sigma, 0.01), 0.5, 2.0)
    prem_tp = ALPHA * fv * vi + beta * fees
    corr_tp = (IL + fees + pay - prem_tp) / V0
    print(f'  Corridor(tp):{corr_tp*100:+.3f}%  (upfront=${ALPHA*fv*vi:.2f}, settl=${beta*fees:.2f}, vi={vi:.2f})')

    # 5. RT v1
    n_c = max(1, int(V0 * 0.30 / nat_cap))
    rt_income = n_c * prem_tp * 0.985
    rt_claim = n_c * pay
    rt = (rt_income - rt_claim) / V0
    print(f'  RT v1:       {rt*100:+.3f}%  ({n_c} certs, income=${rt_income:.2f}, claims=${rt_claim:.2f})')

    # 6. Put spread (BS+IV)
    iv = sigma + IV_PREMIUM
    ps_cost = (V0/S_s) * (bs_put(S_s, S_s, iv, T_WEEK) - bs_put(S_s, pos['barrier'], iv, T_WEEK)) * (1 + OPTION_SPREAD_PCT)
    ps_pay = (V0/S_s) * max(0, max(0, S_s-S_e) - max(0, pos['barrier']-S_e))
    put = (IL + fees + ps_pay - ps_cost) / V0
    print(f'  Put(BS+IV):  {put*100:+.3f}%  (cost=${ps_cost:.2f}, pay=${ps_pay:.2f})')

    # 7. Perp hedge
    delta = cl_delta(S_s, N_LIQ, pos['p_l'], pos['p_u'])
    perp_pnl = -delta * (S_e - S_s)
    perp_fund = abs(delta) * S_s * PERP_FUNDING_APY * T_WEEK
    perp_open_fee = abs(delta) * S_s * PERP_FEE_BPS / 10000
    perp_close_fee = abs(delta) * S_e * PERP_FEE_BPS / 10000
    perp_impact = abs(delta) * (S_s + S_e) / 2 * PERP_IMPACT_BPS / 10000
    perp_total_cost = perp_fund + perp_open_fee + perp_close_fee + perp_impact
    perp = (IL + fees + perp_pnl - perp_total_cost) / V0
    print(f'  Perp(80%):   {perp*100:+.3f}%  (pnl=${perp_pnl:.2f}, cost=${perp_total_cost:.2f})')

    # 8. Corridor + Perp Tail (daily granularity)
    tail_pnl = 0.0; tail_cost = 0.0; tail_open = False; tail_sol = 0.0; tail_entry = 0.0
    for d in range(7):
        S_day = week_prices[d]; S_next = week_prices[min(d+1, 7)]
        if S_day < pos['p_l'] and not tail_open:
            tail_sol = N_LIQ * (1/np.sqrt(pos['p_l']) - 1/np.sqrt(pos['p_u']))
            tail_entry = S_day
            tail_cost += tail_sol * S_day * PERP_FEE_BPS / 10000  # open fee
            tail_open = True
        if tail_open:
            tail_pnl += -tail_sol * (S_next - S_day)
            tail_cost += tail_sol * S_day * PERP_FUNDING_APY / 365
            if S_next >= pos['p_l']:
                tail_cost += tail_sol * S_next * PERP_FEE_BPS / 10000  # close fee (at exit price!)
                tail_cost += tail_sol * S_next * PERP_IMPACT_BPS / 10000
                tail_open = False
    if tail_open:
        tail_cost += tail_sol * S_e * PERP_FEE_BPS / 10000
        tail_cost += tail_sol * S_e * PERP_IMPACT_BPS / 10000
    corr_tail = (IL + fees + pay - prem_tp + tail_pnl - tail_cost) / V0
    triggered = 'YES' if tail_sol > 0 else 'NO'
    print(f'  Corr+Tail:   {corr_tail*100:+.3f}%  (tail_pnl=${tail_pnl:.2f}, tail_cost=${tail_cost:.2f}, triggered={triggered})')

MANUAL SINGLE-WEEK WALKTHROUGH

--- +-5% Width, Week 5 ---
Entry: $173.33, Exit: $175.81, Change: +1.43%
Position: V0=$6506.36, V_end=$6545.12, IL=$38.76 (+0.60%)
Range: [$164.66, $181.99], Barrier=$164.66
Natural cap: $243.05 (3.7% of V0)
In-range: 100%, Fees: $296.04 (4.55%)
Trailing 30d vol: 65.8%
Fair value: $96.08, Payout: $0.00

Strategy returns (% of V0):
  Bond:        +0.218%
  Plain LP:    +5.146%  (IL=+0.60% + fees=+4.55%)
  Corridor(fx):+4.659%  (prem=$105.68, eff=$31.67, pay=$0.00)
  Corridor(tp):+3.465%  (upfront=$42.10, settl=$67.25, vi=1.10)
  RT v1:       +13.243%  (8 certs, income=$861.66, claims=$0.00)
  Put(BS+IV):  +2.698%  (cost=$159.24, pay=$0.00)
  Perp(80%):   +3.605%  (pnl=$-45.47, cost=$54.75)
  Corr+Tail:   +3.465%  (tail_pnl=$0.00, tail_cost=$0.00, triggered=NO)

--- +-10% Width, Week 5 ---
Entry: $173.33, Exit: $175.81, Change: +1.43%
Position: V0=$12882.80, V_end=$12963.89, IL=$81.09 (+0.63%)
Range: [$155.99, $190.66], Barrier=$155.99
Natural cap: $959.38

Fair value: $274.40, Payout: $0.00

Strategy returns (% of V0):
  Bond:        +0.218%
  Plain LP:    +3.779%  (IL=+0.63% + fees=+3.15%)
  Corridor(fx):+2.067%  (prem=$301.84, eff=$220.68, pay=$0.00)
  Corridor(tp):+1.355%  (upfront=$120.23, settl=$192.08, vi=1.10)
  RT v1:       +9.551%  (4 certs, income=$1230.50, claims=$0.00)
  Put(BS+IV):  -0.262%  (cost=$520.70, pay=$0.00)
  Perp(80%):   +2.277%  (pnl=$-87.81, cost=$105.72)
  Corr+Tail:   +1.355%  (tail_pnl=$0.00, tail_cost=$0.00, triggered=NO)


In [5]:
def run_clean_backtest(closes, width_key, wcfg, daily_fee_override=None):
    """Clean backtest: all returns as pct of V0. No modes, no options."""
    w = wcfg['width']
    daily_fee = daily_fee_override if daily_fee_override else wcfg['fee']
    fs_frac = wcfg['fs_bps'] / 10000

    results = {s: [] for s in ['bond', 'plain_lp', 'corridor_fixed', 'corridor_tp',
                                 'corridor_jito', 'rt_v1', 'put_bs_iv', 'perp_80', 'corridor_tail']}
    rt_premiums = 0; rt_claims = 0; rt_claim_weeks = 0

    for wk_start in range(35, len(closes) - 7, 7):
        si = wk_start; ei = si + 7
        if ei >= len(closes): break
        S_s = closes[si]; S_e = closes[ei]
        week_prices = closes[si:ei+1]

        pos = make_position(S_s, w)
        V0 = pos['V0']
        V_end = float(cl_value_vec(np.array([S_e]), N_LIQ, pos['p_l'], pos['p_u'])[0])
        IL = V_end - V0

        in_rng = np.mean((week_prices[1:] >= pos['p_l']) & (week_prices[1:] <= pos['p_u']))
        fees = V0 * daily_fee * 7 * (in_rng * 0.95 + 0.05)

        # Trailing vol
        if si >= 30:
            lr = np.diff(np.log(closes[si-30:si+1]))
            sigma = float(np.std(lr, ddof=1) * np.sqrt(365))
        else:
            sigma = 0.65

        # Vol indicator
        if si >= 7:
            lr7 = np.diff(np.log(closes[max(0,si-7):si+1]))
            s7 = float(np.std(lr7, ddof=1) * np.sqrt(365)) if len(lr7) > 1 else sigma
            vi = float(np.clip(s7 / max(sigma, 0.01), 0.5, 2.0))
        else:
            vi = 1.0

        fv = fv_quadrature(S_s, sigma, pos['barrier'], pos['nat_cap'], N_LIQ, pos['p_l'], pos['p_u'])
        pay = float(corridor_payoff_vec(np.array([S_e]), S_s, pos['barrier'], pos['nat_cap'],
                                         N_LIQ, pos['p_l'], pos['p_u'])[0])

        # -- 1. Bond --
        results['bond'].append(BOND_WK)

        # -- 2. Plain LP --
        results['plain_lp'].append((IL + fees) / V0)

        # -- 3. Corridor (fixed) --
        prem_fx = fv * MARKUP
        eff_fx = max(0, prem_fx - fees * fs_frac)
        results['corridor_fixed'].append((IL + fees + pay - eff_fx) / V0)

        # -- 4. Corridor (two-part) --
        wf_exp = V0 * daily_fee * 7
        beta = max(0, (MARKUP - ALPHA) * fv / max(wf_exp, 1e-10))
        prem_tp = ALPHA * fv * vi + beta * fees
        results['corridor_tp'].append((IL + fees + pay - prem_tp) / V0)

        # -- 5. Corridor + jito --
        jito = V0 * SOL_FRACTION * ((1 + JITO_APY)**(1/52) - 1)
        results['corridor_jito'].append((IL + fees + pay - prem_tp + jito) / V0)

        # -- 6. RT v1 --
        n_c = max(1, int(V0 * 0.30 / max(pos['nat_cap'], 1)))
        rt_inc = n_c * prem_tp * 0.985
        rt_cl = n_c * pay
        results['rt_v1'].append((rt_inc - rt_cl) / V0)
        rt_premiums += rt_inc; rt_claims += rt_cl
        if pay > 0: rt_claim_weeks += 1

        # -- 7. Put spread (BS+IV) --
        iv = sigma + IV_PREMIUM
        ps_cost = (V0/S_s) * (bs_put(S_s, S_s, iv, T_WEEK) - bs_put(S_s, pos['barrier'], iv, T_WEEK)) * (1 + OPTION_SPREAD_PCT)
        ps_pay_v = (V0/S_s) * max(0, max(0, S_s - S_e) - max(0, pos['barrier'] - S_e))
        results['put_bs_iv'].append((IL + fees + ps_pay_v - ps_cost) / V0)

        # -- 8. Perp delta hedge (80%) --
        delta = cl_delta(S_s, N_LIQ, pos['p_l'], pos['p_u'])
        p_pnl = -delta * (S_e - S_s)
        p_fund = abs(delta) * S_s * PERP_FUNDING_APY * T_WEEK
        p_open = abs(delta) * S_s * PERP_FEE_BPS / 10000
        p_close = abs(delta) * S_e * PERP_FEE_BPS / 10000
        p_impact = abs(delta) * (S_s + S_e) / 2 * PERP_IMPACT_BPS / 10000
        results['perp_80'].append((IL + fees + p_pnl - p_fund - p_open - p_close - p_impact) / V0)

        # -- 9. Corridor + Perp Tail --
        t_pnl = 0.0; t_cost = 0.0; t_open = False; t_sol = 0.0; t_entry = 0.0
        for d in range(7):
            S_d = week_prices[d]; S_n = week_prices[min(d+1, 7)]
            if S_d < pos['p_l'] and not t_open:
                t_sol = N_LIQ * (1/np.sqrt(pos['p_l']) - 1/np.sqrt(pos['p_u']))
                t_entry = S_d
                t_cost += t_sol * S_d * PERP_FEE_BPS / 10000  # open fee at entry price
                t_open = True
            if t_open:
                t_pnl += -t_sol * (S_n - S_d)
                t_cost += t_sol * S_d * PERP_FUNDING_APY / 365
                if S_n >= pos['p_l']:
                    t_cost += t_sol * S_n * PERP_FEE_BPS / 10000  # close fee at EXIT price
                    t_cost += t_sol * S_n * PERP_IMPACT_BPS / 10000
                    t_open = False
        if t_open:  # still open at week end
            t_cost += t_sol * S_e * PERP_FEE_BPS / 10000
            t_cost += t_sol * S_e * PERP_IMPACT_BPS / 10000
        results['corridor_tail'].append((IL + fees + pay - prem_tp + t_pnl - t_cost) / V0)

    n_wks = len(results['bond'])
    return results, n_wks, rt_premiums, rt_claims, rt_claim_weeks

print('Clean backtest engine defined.')

Clean backtest engine defined.


In [6]:
STRAT_NAMES = {
    'bond': '1. Bond (12%)',
    'plain_lp': '2. Plain LP',
    'corridor_fixed': '3. Corridor (fixed)',
    'corridor_tp': '4. Corridor (two-part)',
    'corridor_jito': '5. Corridor+jito',
    'rt_v1': '6. RT v1',
    'put_bs_iv': '7. Put Spread (BS+IV)',
    'perp_80': '8. Perp (80%)',
    'corridor_tail': '9. Corridor+PerpTail',
}

for wk_key, wcfg in WIDTHS.items():
    res, n_wks, rt_prem, rt_cl, rt_cl_wks = run_clean_backtest(closes, wk_key, wcfg)

    print(f'\n{"="*95}')
    print(f'{wcfg["label"]} WIDTH ({n_wks} weeks, fee={wcfg["fee"]*100:.2f}%/day)')
    print(f'{"="*95}')
    hdr = f'{"Strategy":<25} {"Med%/wk":>8} {"Mean%":>8} {"Sharpe":>7} {"P(+)":>6} {"5th%":>8} {"AnnMed%":>9}'
    print(hdr)
    print('-'*75)

    for skey, sname in STRAT_NAMES.items():
        a = np.array(res[skey])
        med = np.median(a)*100; mn = np.mean(a)*100
        sh = np.mean(a)/np.std(a) if np.std(a) > 1e-10 else 0
        pp = np.mean(a > 0)*100; p5 = np.percentile(a, 5)*100
        ann = ((1 + np.median(a))**52 - 1)*100
        print(f'{sname:<25} {med:>+7.3f}% {mn:>+7.3f}% {sh:>+6.3f} {pp:>5.1f}% {p5:>+7.2f}% {ann:>+8.1f}%')

    print(f'\n  RT: premiums=${rt_prem:.0f}, claims=${rt_cl:.0f}, loss ratio={rt_cl/max(rt_prem,1)*100:.1f}%')
    print(f'  Claim weeks: {rt_cl_wks}/{n_wks} ({rt_cl_wks/n_wks*100:.0f}%)')


+-5% WIDTH (47 weeks, fee=0.65%/day)
Strategy                   Med%/wk    Mean%  Sharpe   P(+)     5th%   AnnMed%
---------------------------------------------------------------------------
1. Bond (12%)              +0.218%  +0.218% +0.000 100.0%   +0.22%    +12.0%
2. Plain LP                +2.318%  -0.258% -0.044  68.1%  -11.75%   +229.3%
3. Corridor (fixed)        +2.200%  +0.253% +0.053  74.5%   -9.16%   +210.1%
4. Corridor (two-part)     +1.957%  +0.012% +0.003  78.7%   -9.01%   +173.9%
5. Corridor+jito           +2.019%  +0.075% +0.017  78.7%   -8.94%   +182.8%
6. RT v1                   +4.850%  -2.310% -0.168  61.7%  -23.16%  +1074.0%
7. Put Spread (BS+IV)      +1.118%  -0.648% -0.155  66.0%   -9.05%    +78.3%
8. Perp (80%)              +0.720%  -0.499% -0.139  55.3%   -6.48%    +45.2%
9. Corridor+PerpTail       +1.053%  -0.561% -0.132  59.6%   -7.92%    +72.4%

  RT: premiums=$26881, claims=$34004, loss ratio=126.5%
  Claim weeks: 28/47 (60%)



+-10% WIDTH (47 weeks, fee=0.45%/day)
Strategy                   Med%/wk    Mean%  Sharpe   P(+)     5th%   AnnMed%
---------------------------------------------------------------------------
1. Bond (12%)              +0.218%  +0.218% +0.000 100.0%   +0.22%    +12.0%
2. Plain LP                +2.655%  +0.373% +0.070  72.3%   -9.84%   +290.6%
3. Corridor (fixed)        +1.389%  +0.691% +0.261  78.7%   -4.54%   +104.9%
4. Corridor (two-part)     +0.969%  +0.396% +0.161  76.6%   -4.49%    +65.1%
5. Corridor+jito           +1.031%  +0.458% +0.187  76.6%   -4.43%    +70.5%
6. RT v1                   +6.209%  -0.222% -0.018  70.2%  -23.60%  +2192.6%
7. Put Spread (BS+IV)      -0.044%  -0.535% -0.297  48.9%   -4.12%     -2.3%
8. Perp (80%)              +1.519%  +0.138% +0.053  66.0%   -4.60%   +119.0%
9. Corridor+PerpTail       +0.969%  +0.249% +0.081  74.5%   -4.44%    +65.1%

  RT: premiums=$48094, claims=$51214, loss ratio=106.5%
  Claim weeks: 28/47 (60%)


In [7]:
print('='*60)
print('SANITY CHECKS')
print('='*60)

for wk_key, wcfg in WIDTHS.items():
    res, n, _, _, _ = run_clean_backtest(closes, wk_key, wcfg)

    bond = np.array(res['bond'])
    plain = np.array(res['plain_lp'])
    corr = np.array(res['corridor_tp'])
    tail = np.array(res['corridor_tail'])

    checks = []
    # 1. Bond consistency
    bond_ann = ((1 + np.median(bond))**52 - 1)*100
    checks.append(('Bond = +12.0% ann', abs(bond_ann - 12.0) < 0.1))
    checks.append(('Bond med = +0.218%/wk', abs(np.median(bond)*100 - 0.218) < 0.01))

    # 2. Plain LP
    checks.append(('Plain LP median > 0', np.median(plain) > 0))
    checks.append(('Plain LP mean < median', np.mean(plain) < np.median(plain)))

    # 3. Corridor+Tail differs from corridor
    diff = np.sum(np.abs(np.array(res['corridor_tail']) - np.array(res['corridor_tp'])) > 1e-8)
    checks.append(('Corr+Tail differs from Corr', diff > 0))

    # 4. Weekly bounds
    all_rets = np.concatenate([np.array(res[s]) for s in res if s != 'bond'])
    checks.append(('All weekly |ret| < 50%', np.all(np.abs(all_rets) < 0.50)))

    print(f'\n  {wcfg["label"]}:')
    for name, passed in checks:
        status = 'PASS' if passed else 'FAIL'
        print(f'    {status} -- {name}')

SANITY CHECKS



  +-5%:
    PASS -- Bond = +12.0% ann
    PASS -- Bond med = +0.218%/wk
    PASS -- Plain LP median > 0
    PASS -- Plain LP mean < median
    PASS -- Corr+Tail differs from Corr
    PASS -- All weekly |ret| < 50%



  +-10%:
    PASS -- Bond = +12.0% ann
    PASS -- Bond med = +0.218%/wk
    PASS -- Plain LP median > 0
    PASS -- Plain LP mean < median
    PASS -- Corr+Tail differs from Corr
    PASS -- All weekly |ret| < 50%


In [8]:
print('='*95)
print('FEE SENSITIVITY ANALYSIS')
print('='*95)

fee_sweep_results = {}
for wk_key, wcfg in WIDTHS.items():
    print(f'\n--- {wcfg["label"]} ---')
    hdr = f'{"Fee/d":>8}'
    for skey in STRAT_NAMES:
        hdr += f' {skey[:8]:>8}'
    hdr += f' {"RT viable":>10}'
    print(hdr)
    print('-'*110)

    for daily_fee in FEE_SCENARIOS:
        res, n, rp, rc, _ = run_clean_backtest(closes, wk_key, wcfg, daily_fee_override=daily_fee)
        fee_sweep_results[(wk_key, daily_fee)] = res

        row = f'{daily_fee*100:>7.2f}%'
        for skey in STRAT_NAMES:
            med = np.median(res[skey]) * 100
            row += f' {med:>+7.2f}%'

        rt_viable = 'YES' if np.median(res['rt_v1']) > 0 and np.mean(res['rt_v1']) > 0 else 'NO'
        row += f' {rt_viable:>10}'
        print(row)

FEE SENSITIVITY ANALYSIS

--- +-5% ---
   Fee/d     bond plain_lp corridor corridor corridor    rt_v1 put_bs_i  perp_80 corridor  RT viable
--------------------------------------------------------------------------------------------------------------


   0.20%   +0.22%   +0.55%   -0.25%   -0.18%   -0.12%   +4.85%   -0.79%   -1.15%   -0.60%         NO


   0.30%   +0.22%   +1.13%   +0.27%   +0.33%   +0.40%   +4.85%   -0.34%   -0.73%   -0.09%         NO


   0.45%   +0.22%   +1.79%   +1.11%   +1.05%   +1.12%   +4.85%   +0.18%   -0.09%   +0.67%         NO


   0.65%   +0.22%   +2.32%   +2.20%   +1.96%   +2.02%   +4.85%   +1.12%   +0.72%   +1.05%         NO


...

   0.20%   +0.22%   +0.90%   -0.71%   -0.78%   -0.72%   +6.21%   -1.71%   -0.14%   -0.78%         NO


   0.30%   +0.22%   +1.60%   +0.13%   -0.08%   -0.02%   +6.21%   -1.09%   +0.47%   -0.08%         NO


   0.45%   +0.22%   +2.65%   +1.39%   +0.97%   +1.03%   +6.21%   -0.04%   +1.52%   +0.97%         NO


   0.65%   +0.22%   +4.05%   +3.07%   +2.37%   +2.43%   +6.21%   +1.22%   +2.77%   +2.37%         NO


   1.00%   +0.22%   +6.39%   +5.99%   +4.78%   +4.84%   +6.21%   +3.43%   +5.22%   +4.78%         NO


In [9]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
strat_colors = {'plain_lp': 'steelblue', 'corridor_tp': 'seagreen',
                'corridor_fixed': 'darkgreen', 'put_bs_iv': 'brown',
                'perp_80': 'teal', 'rt_v1': 'darkorange', 'corridor_tail': 'purple'}

for col, (wk_key, wcfg) in enumerate(WIDTHS.items()):
    # Top: median return
    ax = axes[0, col]
    for skey, color in strat_colors.items():
        meds = [np.median(fee_sweep_results[(wk_key, f)][skey])*100 for f in FEE_SCENARIOS]
        ax.plot([f*100 for f in FEE_SCENARIOS], meds, '-o', color=color, lw=2, ms=5,
                label=STRAT_NAMES[skey][:15])
    ax.axhline(y=0, color='black', lw=1, ls='--')
    ax.axhline(y=BOND_WK*100, color='gray', lw=1, ls=':', label='Bond')
    ax.axvline(x=wcfg['fee']*100, color='gray', lw=1.5, ls=':', alpha=0.5)
    ax.set_xlabel('Daily Fee Rate (%)')
    ax.set_ylabel('Median Weekly Return (%)')
    ax.set_title(f'{wcfg["label"]} -- Median Return')
    ax.legend(fontsize=6, ncol=2)

    # Bottom: Sharpe
    ax = axes[1, col]
    for skey, color in strat_colors.items():
        sharpes = []
        for f in FEE_SCENARIOS:
            a = np.array(fee_sweep_results[(wk_key, f)][skey])
            sharpes.append(np.mean(a)/np.std(a) if np.std(a) > 1e-10 else 0)
        ax.plot([f*100 for f in FEE_SCENARIOS], sharpes, '-o', color=color, lw=2, ms=5,
                label=STRAT_NAMES[skey][:15])
    ax.axhline(y=0, color='black', lw=1, ls='--')
    ax.axvline(x=wcfg['fee']*100, color='gray', lw=1.5, ls=':', alpha=0.5)
    ax.set_xlabel('Daily Fee Rate (%)')
    ax.set_ylabel('Sharpe Ratio')
    ax.set_title(f'{wcfg["label"]} -- Sharpe')
    ax.legend(fontsize=6, ncol=2)

plt.tight_layout()
os.makedirs('data', exist_ok=True)
plt.savefig('data/verify_fee_sensitivity.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: data/verify_fee_sensitivity.png')

Saved: data/verify_fee_sensitivity.png


In [10]:
for wk_key, wcfg in WIDTHS.items():
    res, n, rp, rc, rcw = run_clean_backtest(closes, wk_key, wcfg)
    rt = np.array(res['rt_v1'])

    print(f'\n{"="*60}')
    print(f'RT DEEP DIVE -- {wcfg["label"]}')
    print(f'{"="*60}')
    print(f'  Premiums: ${rp:.0f}, Claims: ${rc:.0f}, Loss ratio: {rc/max(rp,1)*100:.1f}%')
    print(f'  Claim weeks: {rcw}/{n} ({rcw/n*100:.0f}%)')
    print(f'  Median weekly: {np.median(rt)*100:+.3f}%')
    print(f'  Mean weekly:   {np.mean(rt)*100:+.3f}%')
    print(f'  Sharpe:        {np.mean(rt)/np.std(rt):.3f}')
    print(f'  P(+):          {np.mean(rt>0)*100:.1f}%')
    print(f'  Worst week:    {np.min(rt)*100:+.2f}%')
    print(f'  Best week:     {np.max(rt)*100:+.2f}%')


RT DEEP DIVE -- +-5%
  Premiums: $26881, Claims: $34004, Loss ratio: 126.5%
  Claim weeks: 28/47 (60%)
  Median weekly: +4.850%
  Mean weekly:   -2.310%
  Sharpe:        -0.168
  P(+):          61.7%
  Worst week:    -26.88%
  Best week:     +13.70%



RT DEEP DIVE -- +-10%
  Premiums: $48094, Claims: $51214, Loss ratio: 106.5%
  Claim weeks: 28/47 (60%)
  Median weekly: +6.209%
  Mean weekly:   -0.222%
  Sharpe:        -0.018
  P(+):          70.2%
  Worst week:    -27.41%
  Best week:     +12.97%


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for col, (wk_key, wcfg) in enumerate(WIDTHS.items()):
    res, n, _, _, _ = run_clean_backtest(closes, wk_key, wcfg)
    ax = axes[col]
    for skey, color in [('corridor_tp', 'seagreen'), ('put_bs_iv', 'brown'),
                         ('perp_80', 'teal'), ('plain_lp', 'steelblue'),
                         ('corridor_tail', 'purple')]:
        wealth = np.cumprod(1 + np.array(res[skey]))
        ax.plot(wealth, color=color, lw=1.5, label=STRAT_NAMES[skey][:15])
    bond_wealth = np.cumprod(1 + np.array(res['bond']))
    ax.plot(bond_wealth, color='black', lw=1, ls='--', label='Bond')
    ax.axhline(y=1, color='gray', lw=0.5)
    ax.set_xlabel('Week')
    ax.set_ylabel('Wealth (1=initial)')
    ax.set_title(f'{wcfg["label"]} -- Cumulative Wealth')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('data/verify_wealth_paths.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: data/verify_wealth_paths.png')

Saved: data/verify_wealth_paths.png


In [12]:
def run_adaptive_backtest(closes, width_key, wcfg, iv_rv_fn, floor=1.05):
    """Same as clean backtest but with adaptive markup."""
    w = wcfg['width']; daily_fee = wcfg['fee']; fs_frac = wcfg['fs_bps']/10000
    res = {s: [] for s in ['corridor_adapt', 'rt_adapt']}

    for wk_start in range(35, len(closes)-7, 7):
        si = wk_start; ei = si+7
        if ei >= len(closes): break
        S_s = closes[si]; S_e = closes[ei]
        wp = closes[si:ei+1]
        pos = make_position(S_s, w)
        V0 = pos['V0']
        V_end = float(cl_value_vec(np.array([S_e]), N_LIQ, pos['p_l'], pos['p_u'])[0])
        IL = V_end - V0
        in_rng = np.mean((wp[1:] >= pos['p_l']) & (wp[1:] <= pos['p_u']))
        fees = V0 * daily_fee * 7 * (in_rng * 0.95 + 0.05)
        if si >= 30:
            lr = np.diff(np.log(closes[si-30:si+1]))
            sigma = float(np.std(lr, ddof=1)*np.sqrt(365))
        else:
            sigma = 0.65
        if si >= 7:
            lr7 = np.diff(np.log(closes[max(0,si-7):si+1]))
            s7 = float(np.std(lr7, ddof=1)*np.sqrt(365)) if len(lr7) > 1 else sigma
            vi = float(np.clip(s7/max(sigma,0.01), 0.5, 2.0))
        else:
            vi = 1.0

        fv = fv_quadrature(S_s, sigma, pos['barrier'], pos['nat_cap'], N_LIQ, pos['p_l'], pos['p_u'])
        pay = float(corridor_payoff_vec(np.array([S_e]), S_s, pos['barrier'], pos['nat_cap'],
                                         N_LIQ, pos['p_l'], pos['p_u'])[0])

        # Adaptive markup
        stress = s7/max(sigma, 0.01) > 1.5 if si >= 7 else False
        iv_rv = iv_rv_fn(s7 if si >= 7 else sigma, sigma, stress)
        eff_markup = max(floor, iv_rv)

        wf_exp = V0 * daily_fee * 7
        beta = max(0, (eff_markup - ALPHA) * fv / max(wf_exp, 1e-10))
        prem = ALPHA * fv * vi + beta * fees
        n_c = max(1, int(V0 * 0.30 / max(pos['nat_cap'], 1)))

        res['corridor_adapt'].append((IL + fees + pay - prem) / V0)
        res['rt_adapt'].append((n_c * prem * 0.985 - n_c * pay) / V0)

    return res

# Test
scenarios = {
    'constant 1.25': lambda s7, s30, st: 1.25,
    'regime switch': lambda s7, s30, st: 1.40 if st or s7/max(s30,0.01) > 1.3 else 1.15,
}

print('='*90)
print('IV/RV ADAPTIVE MARKUP vs FIXED')
print('='*90)

for wk_key, wcfg in WIDTHS.items():
    res_fixed, n, _, _, _ = run_clean_backtest(closes, wk_key, wcfg)
    corr_f = np.array(res_fixed['corridor_tp'])
    rt_f = np.array(res_fixed['rt_v1'])

    print(f'\n  {wcfg["label"]}:')
    hdr = f'  {"Config":<30} {"LP Med":>8} {"LP Sharpe":>10} {"RT Med":>8} {"RT Mean":>9}'
    print(hdr)
    print(f'  {"-"*70}')
    sh_f = np.mean(corr_f)/np.std(corr_f) if np.std(corr_f) > 1e-10 else 0
    print(f'  {"Fixed 1.10x":<30} {np.median(corr_f)*100:>+7.3f}% {sh_f:>9.3f} '
          f'{np.median(rt_f)*100:>+7.3f}% {np.mean(rt_f)*100:>+8.3f}%')

    for sname, sfn in scenarios.items():
        res_a = run_adaptive_backtest(closes, wk_key, wcfg, sfn)
        ca = np.array(res_a['corridor_adapt']); ra = np.array(res_a['rt_adapt'])
        sh_a = np.mean(ca)/np.std(ca) if np.std(ca) > 1e-10 else 0
        print(f'  {"Adaptive " + sname:<30} {np.median(ca)*100:>+7.3f}% {sh_a:>9.3f} '
              f'{np.median(ra)*100:>+7.3f}% {np.mean(ra)*100:>+8.3f}%')

IV/RV ADAPTIVE MARKUP vs FIXED



  +-5%:
  Config                           LP Med  LP Sharpe   RT Med   RT Mean
  ----------------------------------------------------------------------
  Fixed 1.10x                     +1.957%     0.003  +4.850%   -2.310%


  Adaptive constant 1.25          +1.830%    -0.028  +6.128%   -1.243%


  Adaptive regime switch          +1.741%    -0.018  +5.479%   -1.592%



  +-10%:
  Config                           LP Med  LP Sharpe   RT Med   RT Mean
  ----------------------------------------------------------------------
  Fixed 1.10x                     +0.969%     0.161  +6.209%   -0.222%


  Adaptive constant 1.25          +0.644%     0.045  +7.254%   +0.907%


  Adaptive regime switch          +0.860%     0.077  +6.557%   +0.578%


## Conclusions

This verification notebook independently confirms:
1. All strategy returns are in % of V0 (position value), correctly normalized
2. Bond returns are identical across widths (+0.218%/wk = +12.0% ann)
3. The perp tail closure fee bug has been fixed -- separate per-leg fees
4. Fee sensitivity shows the breakeven daily rate for each strategy
5. Corridor+PerpTail now correctly differs from the plain corridor